## Lab | MCP

### Step 1: Setup & Installation

In [2]:
%pip install -qU langchain langchain-openai langchain-mcp-adapters langgraph mcp python-dotenv "pydantic>=2,<3"

Note: you may need to restart the kernel to use updated packages.


In [36]:
import os
import shutil
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Add it to your .env file.")


GITHUB_TOKEN = os.getenv("GITHUB_PERSONAL_ACCESS_TOKEN")      # only needed for the GitHub server
SAMPLE_DOCS_DIR = os.path.join(os.getcwd(), "sample_docs")   # the ONLY folder the Filesystem server may see
# Using other servers? Load their secrets here the same way, e.g. NOTION_TOKEN = os.getenv("NOTION_TOKEN")

print("OpenAI key:   loaded")
print("Node.js npx: ", "found" if shutil.which("npx") else "NOT found: needed only for servers started with npx")
print("GitHub token:", "loaded" if GITHUB_TOKEN else "missing: needed only if you use the GitHub server")
print("sample_docs: ", "found" if os.path.isdir(SAMPLE_DOCS_DIR) else "NOT found: put sample_docs/ next to this notebook")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)    


OpenAI key:   loaded
Node.js npx:  found
GitHub token: loaded
sample_docs:  found


In [37]:
import sys

if sys.platform == "win32":
    import mcp.client.stdio as _stdio
    _stdio.stdio_client.__wrapped__.__defaults__ = (open(os.devnull, "w"),)

In [38]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

async def log_tool_calls(request, handler):
    # BEFORE: which server, which tool, which inputs
    print(f"[LOG] {request.server_name} -> {request.name}  {request.args}")
    result = await handler(request)   # the real tool runs on the server
    # AFTER: what the server sent back
    text = " ".join(getattr(part, "text", str(part)) for part in result.content)
    print(f"[LOG]    returned: {text[:150]}...")
    return result

### Step 2: Connect to MCP Servers

In [39]:
filesystem_client = MultiServerMCPClient(
    {
        "filesystem": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", SAMPLE_DOCS_DIR],
        },
    },    
    tool_interceptors=[log_tool_calls],
)

fs_tools = await filesystem_client.get_tools()
print([t.name for t in fs_tools])
print(f"Total tools: {len(fs_tools)}")

['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']
Total tools: 14


In [40]:
github_client = MultiServerMCPClient(
    {
        "github": {
            "transport": "streamable_http",
            "url": "https://api.githubcopilot.com/mcp/",
            "headers": {"Authorization": f"Bearer {GITHUB_TOKEN}"},
        }, 
    },    
    tool_interceptors=[log_tool_calls],
)

github_tools= await github_client.get_tools()
github_tools = await github_client.get_tools()
print([t.name for t in github_tools])
print(f"Total tools: {len(github_tools)}")

['add_comment_to_pending_review', 'add_issue_comment', 'add_reply_to_pull_request_comment', 'create_branch', 'create_or_update_file', 'create_pull_request', 'create_repository', 'delete_file', 'fork_repository', 'get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_me', 'get_release_by_tag', 'get_tag', 'get_team_members', 'get_teams', 'issue_read', 'issue_write', 'list_branches', 'list_commits', 'list_issue_fields', 'list_issue_types', 'list_issues', 'list_pull_requests', 'list_releases', 'list_repository_collaborators', 'list_tags', 'merge_pull_request', 'pull_request_read', 'pull_request_review_write', 'push_files', 'request_copilot_review', 'run_secret_scanning', 'search_code', 'search_commits', 'search_issues', 'search_pull_requests', 'search_repositories', 'search_users', 'sub_issue_write', 'ui_get', 'update_issue_comment', 'update_pull_request', 'update_pull_request_branch']
Total tools: 46


In [42]:
clients = {
    "filesystem": filesystem_client,
    "github": github_client,
}

resources_by_server = {}

for name, c in clients.items():
    try:
        resources = await c.get_resources()
        resources_by_server[name] = resources
        if resources:
            print(f"{name}: {len(resources)} resources")
            print(f"   first: {resources[0].as_string()[:200]}")
        else:
            print(f"{name}: no resources")
    except Exception as e:
        resources_by_server[name] = []
        print(f"{name}: none or not supported ({type(e).__name__}: {e})")

filesystem: none or not supported (ExceptionGroup: unhandled errors in a TaskGroup (1 sub-exception))
github: 4 resources
   first: <!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <meta http-equiv="Content-Security-Policy" conte



 Server | Transport |	Secret
 --- | --- | --- 
github	| HTTP	| in headers (loaded from .env via os.getenv, never hard-coded) 
filesystem|	stdio|	none (only the allowed folder SAMPLE_DOCS_DIR as an argument) 

### Step 3: Load MCP Tools into LangChain

Note: `get_tools()` already converts every MCP tool into a LangChain tool.


In [44]:
# Load tools from the MCP server
# The client is stateless: get_tools() creates ephemeral sessions under the hood

tool_groups = {
    "filesystem": fs_tools,
    "github": github_tools,
}

total = sum(len(t) for t in tool_groups.values())
print(f"Loaded {total} tools from {len(tool_groups)} MCP servers")

for server, server_tools in tool_groups.items():
    print(f"\n{server} ({len(server_tools)} tools):")
    for tool in server_tools:
        desc = (tool.description or "")[:80]
        print(f"  - {tool.name}: {desc}...")

Loaded 60 tools from 2 MCP servers

filesystem (14 tools):
  - read_file: Read the complete contents of a file as text. DEPRECATED: Use read_text_file ins...
  - read_text_file: Read the complete contents of a file from the file system as text. Handles vario...
  - read_media_file: Read a file and return it as a base64-encoded content block with its MIME type. ...
  - read_multiple_files: Read the contents of multiple files simultaneously. This is more efficient than ...
  - write_file: Create a new file or completely overwrite an existing file with new content. Use...
  - edit_file: Make line-based edits to a text file. Each edit replaces exact line sequences wi...
  - create_directory: Create a new directory or ensure a directory exists. Can create multiple nested ...
  - list_directory: Get a detailed listing of all files and directories in a specified path. Results...
  - list_directory_with_sizes: Get a detailed listing of all files and directories in a specified path, includi...


### Step 4: Create Agent with MCP Tools 

In [47]:
fs_agent= create_agent(
            model=llm,
            tools=fs_tools,
            system_prompt=("You are a helpful assistant that answers questions about MCP"
                        f"and steps using the files in the folder {SAMPLE_DOCS_DIR}. "
                        "Use the filesystem tools to list and read the files before answering, "
                        "and say so if the answer is not in the files."
            )
        )

print(f"Agent created with {len(fs_tools)} MCP tools")
print(f"Tools: {[tool.name for tool in fs_tools]}")

Agent created with 14 MCP tools
Tools: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']


In [49]:
fs_question = "What are the steps to install an MCP server, according to the docs?"

fs_result = await fs_agent.ainvoke({"messages": [HumanMessage(fs_question)]}
)
print(fs_result["messages"][-1].content)

[LOG] filesystem -> list_directory  {'path': '/home/fatmamerayy/AI-Consultant-Bootcamp/Week 4/lab-mcp-langchain/sample_docs'}
[LOG]    returned: [FILE] Lab_agent_speak_fluently.md
[FILE] mcp_introduction.md...
[LOG] filesystem -> read_text_file  {'path': '/home/fatmamerayy/AI-Consultant-Bootcamp/Week 4/lab-mcp-langchain/sample_docs/Lab_agent_speak_fluently.md'}
[LOG] filesystem -> read_text_file  {'path': '/home/fatmamerayy/AI-Consultant-Bootcamp/Week 4/lab-mcp-langchain/sample_docs/mcp_introduction.md'}
[LOG]    returned: # MCP Introduction with LangChain

This notebook demonstrates connecting to MCP servers and integrating them with LangChain agents using the `langchai...
[LOG]    returned: # LAB: Agents that Speak MCP Fluently

## Objective

In `mcp_introduction.ipynb` we connected an agent to **one remote** MCP server. In this lab we go...
The steps to install an MCP (Model Context Protocol) server, as outlined in the documentation, are as follows:

### Step 1: Setup and Installati

In [50]:
github_agent= create_agent(
            model=llm,
            tools=github_tools,
            system_prompt=("You are a helpful assistant that answers questions about GitHub repositories. "
                        "Use the GitHub tools to read files (such as the README) before answering. "
                        "Base your answer only on what the tools return, and say so if you can't find it."
            )
        )

print(f"Agent created with {len(github_tools)} MCP tools")
print(f"Tools: {[tool.name for tool in github_tools]}")


git_question=(
    "Read the README of the GitHub repository langchain-ai/langchain-mcp-adapters. "
    "Which MCP transports does it show in its examples?"
)

git_result=await github_agent.ainvoke({"messages":[HumanMessage(content=git_question)]})
print(git_result["messages"][-1].content)

Agent created with 46 MCP tools
Tools: ['add_comment_to_pending_review', 'add_issue_comment', 'add_reply_to_pull_request_comment', 'create_branch', 'create_or_update_file', 'create_pull_request', 'create_repository', 'delete_file', 'fork_repository', 'get_commit', 'get_file_contents', 'get_label', 'get_latest_release', 'get_me', 'get_release_by_tag', 'get_tag', 'get_team_members', 'get_teams', 'issue_read', 'issue_write', 'list_branches', 'list_commits', 'list_issue_fields', 'list_issue_types', 'list_issues', 'list_pull_requests', 'list_releases', 'list_repository_collaborators', 'list_tags', 'merge_pull_request', 'pull_request_read', 'pull_request_review_write', 'push_files', 'request_copilot_review', 'run_secret_scanning', 'search_code', 'search_commits', 'search_issues', 'search_pull_requests', 'search_repositories', 'search_users', 'sub_issue_write', 'ui_get', 'update_issue_comment', 'update_pull_request', 'update_pull_request_branch']
[LOG] github -> get_file_contents  {'owner': '

### Step 5: Access MCP Resources

In [51]:
docs_client= MultiServerMCPClient(
    {"langchain-docs": {
            "transport": "http",
            "url": "https://docs.langchain.com/mcp"
    },
    },    
    tool_interceptors=[log_tool_calls],
)

print("MCP client configured for LangChain documentation server")
print(f"Server: langchain-docs → https://docs.langchain.com/mcp")

MCP client configured for LangChain documentation server
Server: langchain-docs → https://docs.langchain.com/mcp


In [52]:
docs_resources=await docs_client.get_resources()
first_text=docs_resources[0].as_string()
print("First resource",first_text)

First resource ---
name: langchain
description: Use when building AI agents, integrating language models with
  tools, orchestrating multi-step workflows, or deploying production agent
  systems. Reach for this skill when working with LangChain, LangGraph, Deep
  Agents, or LangSmith for agent development, testing, and deployment.
metadata:
  mintlify-proj: langchain
  version: "1.0"
---

# LangChain Skill

## Product summary

LangChain is an open-source framework for building AI agents and LLM applications. It provides three core layers: **LangChain** (high-level agent harness with `create_agent`), **LangGraph** (low-level orchestration runtime for stateful workflows), and **Deep Agents** (batteries-included harness with planning, filesystem, subagents, and memory). Use LangChain's `create_agent` for customizable agents, LangGraph for deterministic + agentic workflows, or Deep Agents for complex multi-step tasks. Deploy with LangSmith Cloud for managed infrastructure or self-host with

In [53]:
docs_tools= await docs_client.get_tools()

guide_agent = create_agent(
    model=llm,
    tools=docs_tools,
    system_prompt=(
        "You are a helpful assistant that answers questions about LangChain. "
        "Use the following document, which is the first resource, to answer the question, "
        "in addition to the tools.\n\n"
        f"--- FIRST RESOURCE ---\n{first_text}\n--- END RESOURCE ---\n\n"
        "Base your answer only on this resource and what the tools return, "
        "and say not in the guide if you can't find the answer."
    )
)

In [54]:
docs_question = "What do I need to add to an agent so that it remembers the conversation between calls?"

docs_result=await guide_agent.ainvoke({"messages":[HumanMessage(content=docs_question)]})
print(docs_result)


{'messages': [HumanMessage(content='What do I need to add to an agent so that it remembers the conversation between calls?', additional_kwargs={}, response_metadata={}, id='541b5cf8-179f-4254-b4f5-1689a55bf83b'), AIMessage(content='To enable an agent to remember the conversation between calls, you need to pass a `thread_id` in the configuration when invoking the agent. This allows the agent to persist conversation history across multiple interactions.\n\nHere’s how you can do it:\n\nWhen invoking the agent, include the `thread_id` in the `config` parameter like this:\n\n```python\nagent.invoke({"messages": [...]}, config={"configurable": {"thread_id": "..."}})\n```\n\nMake sure to replace `"..."` with the actual thread ID you want to use for the conversation. This will ensure that the agent retains the context of the conversation across calls.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 3925, 'total_tokens': 4053,

### Step 6: Build a Complete MCP-Enabled Agent 

In [57]:
all_tools= fs_tools + github_tools

for label, group in [("filesystem", fs_tools), ("github", github_tools)]:
    print(f"\n{label} ({len(group)} tools):")
    print("  " + ", ".join(t.name for t in group))


filesystem (14 tools):
  read_file, read_text_file, read_media_file, read_multiple_files, write_file, edit_file, create_directory, list_directory, list_directory_with_sizes, directory_tree, move_file, search_files, get_file_info, list_allowed_directories

github (46 tools):
  add_comment_to_pending_review, add_issue_comment, add_reply_to_pull_request_comment, create_branch, create_or_update_file, create_pull_request, create_repository, delete_file, fork_repository, get_commit, get_file_contents, get_label, get_latest_release, get_me, get_release_by_tag, get_tag, get_team_members, get_teams, issue_read, issue_write, list_branches, list_commits, list_issue_fields, list_issue_types, list_issues, list_pull_requests, list_releases, list_repository_collaborators, list_tags, merge_pull_request, pull_request_read, pull_request_review_write, push_files, request_copilot_review, run_secret_scanning, search_code, search_commits, search_issues, search_pull_requests, search_repositories, search_use

In [56]:
combined_agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=(
        "You are a helpful assistant with access to local files and GitHub. "
        f"Local files are in {SAMPLE_DOCS_DIR}. "
        "Use the filesystem tools for local files and the GitHub tools for repositories. "
        "Base your answers only on what the tools return, and say so if you can't find something."
    ),
)


question = (
    "Read the MCP lesson file in sample_docs and find which MCP transports it mentions. "
    "Then read the README of the GitHub repository langchain-ai/langchain-mcp-adapters "
    "and tell me which of those transports its examples show."
)


result = await combined_agent.ainvoke({"messages": [HumanMessage(content=question)]})



# Names of the tools the agent called, in order
called = [
    call["name"]
    for msg in result["messages"]
    for call in getattr(msg, "tool_calls", [])
]
print("\nTools called:", called)

print("\nAnswer:\n", result["messages"][-1].content)


[LOG] filesystem -> read_text_file  {'path': '/home/fatmamerayy/AI-Consultant-Bootcamp/Week 4/lab-mcp-langchain/sample_docs/mcp_lesson.txt'}
[LOG]    returned: ENOENT: no such file or directory, open '/home/fatmamerayy/AI-Consultant-Bootcamp/Week 4/lab-mcp-langchain/sample_docs/mcp_lesson.txt'...
[LOG] filesystem -> list_directory  {'path': '/home/fatmamerayy/AI-Consultant-Bootcamp/Week 4/lab-mcp-langchain/sample_docs'}
[LOG]    returned: [FILE] Lab_agent_speak_fluently.md
[FILE] mcp_introduction.md...
[LOG] filesystem -> read_text_file  {'path': '/home/fatmamerayy/AI-Consultant-Bootcamp/Week 4/lab-mcp-langchain/sample_docs/Lab_agent_speak_fluently.md'}
[LOG] filesystem -> read_text_file  {'path': '/home/fatmamerayy/AI-Consultant-Bootcamp/Week 4/lab-mcp-langchain/sample_docs/mcp_introduction.md'}
[LOG]    returned: # MCP Introduction with LangChain

This notebook demonstrates connecting to MCP servers and integrating them with LangChain agents using the `langchai...
[LOG]    returned: 